# 02 · Asynchronous Programming: `asyncio`, `async` & `await`

**Module 13 – Concurrency** · Estimated time: 60 min

> 📘 The handbook's Section 10 intro promises "asynchronous programming" but never covers it. The 📗 bootcamp doesn't either. This lesson fills that gap.

### 🎯 What you will learn
- What async I/O is and when it beats threads
- Coroutines: `async def` and `await`
- Running coroutines: `asyncio.run()` (and top-level `await` in Jupyter)
- Running things concurrently: `gather`, `create_task`, ✅ `TaskGroup` (3.11)
- Timeouts (`asyncio.timeout`, 3.11), cancellation, error handling
- Limiting concurrency with `Semaphore`; producer/consumer with `asyncio.Queue`
- Async iteration and context managers (`async for`, `async with`)
- Mixing in blocking code with `asyncio.to_thread`

## 1. The idea: one thread, many waiting tasks

A program that downloads 100 web pages spends most of its time **waiting**. `asyncio` runs an **event loop** on a single thread: whenever a task hits an `await` on something slow, the loop switches to another task that's ready. That's *cooperative multitasking*.

```
Task A: ──request──[ waiting ...... ]──process──
Task B:     ──request──[ waiting ...... ]──process──
Task C:         ──request──[ waiting ...... ]──process──
              (one thread switches between them at each await)
```

Great for: web servers (FastAPI), scrapers, chat bots, many simultaneous network connections.

## 2. Coroutines: `async def` and `await`

In [1]:
import asyncio
import time

async def greet(name, delay):
    await asyncio.sleep(delay)        # non-blocking sleep: lets other tasks run
    print(f"Hello, {name}! (after {delay}s)")
    return name.upper()

coro = greet("Asha", 1)
print(type(coro))                     # calling an async function returns a COROUTINE object; nothing runs yet
coro.close()                          # (tidy up the unused coroutine)

<class 'coroutine'>


To actually run coroutines:
- In a **script**: `asyncio.run(main())`, the standard entry point
- In **Jupyter/IPython**: an event loop is already running, so use top-level `await main()` (calling `asyncio.run` there raises an error)

```python
# script.py
import asyncio

async def main():
    result = await greet("Asha", 1)
    print(result)

if __name__ == "__main__":
    asyncio.run(main())
```

In [2]:
result = await greet("Asha", 1)      # top-level await works in notebooks
print(result)

Hello, Asha! (after 1s)
ASHA


## 3. Sequential vs concurrent

`await`ing one after another is **sequential**: no speed-up!

In [3]:
async def fetch(item, delay=1):
    print(f"start {item}")
    await asyncio.sleep(delay)        # pretend: network I/O
    print(f"done  {item}")
    return f"{item}-data"

start = time.perf_counter()
a = await fetch("A")
b = await fetch("B")
c = await fetch("C")
print(f"sequential: {time.perf_counter() - start:.1f}s", [a, b, c])

start A


done  A
start B


done  B
start C


done  C
sequential: 3.0s ['A-data', 'B-data', 'C-data']


### `asyncio.gather`: run several coroutines concurrently

In [4]:
start = time.perf_counter()
results = await asyncio.gather(fetch("A"), fetch("B"), fetch("C"))
print(f"gather: {time.perf_counter() - start:.1f}s", results)     # ≈1s, results in order

start A
start B
start C


done  A
done  B
done  C
gather: 1.0s ['A-data', 'B-data', 'C-data']


### Tasks: start now, await later

In [5]:
start = time.perf_counter()
task1 = asyncio.create_task(fetch("X", 1))       # scheduled immediately
task2 = asyncio.create_task(fetch("Y", 1.5))
print("tasks created, doing other work...")
await asyncio.sleep(0.5)
print("other work done, now waiting for tasks")
print(await task1, await task2, f"{time.perf_counter() - start:.1f}s")

tasks created, doing other work...
start X
start Y


other work done, now waiting for tasks


done  X


done  Y
X-data Y-data 1.5s


## 4. ✅ `TaskGroup` (3.11+): structured concurrency

The modern, safer replacement for most uses of `gather`/`create_task`: all tasks are awaited when the block exits, and **if one fails, the others are cancelled** and the errors are raised as an `ExceptionGroup` (Module 07).

In [6]:
async def main():
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(fetch("page1", 0.5))
        t2 = tg.create_task(fetch("page2", 1.0))
        t3 = tg.create_task(fetch("page3", 0.7))
    # here, all tasks are guaranteed to be finished
    return [t1.result(), t2.result(), t3.result()]

print(await main())

start page1
start page2
start page3


done  page1
done  page3


done  page2
['page1-data', 'page2-data', 'page3-data']


In [7]:
async def might_fail(n):
    await asyncio.sleep(0.2 * n)
    if n == 2:
        raise ValueError(f"task {n} failed")
    return n

try:
    async with asyncio.TaskGroup() as tg:
        for n in range(1, 5):
            tg.create_task(might_fail(n))
except* ValueError as eg:
    print("Errors:", [str(e) for e in eg.exceptions])

Errors: ['task 2 failed']


## 5. Timeouts and cancellation

In [8]:
async def slow_operation():
    await asyncio.sleep(5)
    return "finished"

try:
    async with asyncio.timeout(1):            # 3.11+
        await slow_operation()
except TimeoutError:
    print("Timed out after 1s")

try:
    await asyncio.wait_for(slow_operation(), timeout=0.5)    # older but still common
except TimeoutError:
    print("wait_for timed out too")

Timed out after 1s


wait_for timed out too


In [9]:
async def long_job():
    try:
        while True:
            await asyncio.sleep(0.3)
            print("working...")
    except asyncio.CancelledError:
        print("cleanup after cancel")
        raise                                  # ✅ always re-raise CancelledError

task = asyncio.create_task(long_job())
await asyncio.sleep(1)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("task cancelled")

working...


working...


working...
cleanup after cancel
task cancelled


## 6. Limiting concurrency: `Semaphore`

Don't open 10,000 connections at once (servers will block you). Limit it:

In [10]:
sem = asyncio.Semaphore(3)              # at most 3 at a time

async def polite_fetch(i):
    async with sem:
        await asyncio.sleep(0.5)
        return i

start = time.perf_counter()
results = await asyncio.gather(*(polite_fetch(i) for i in range(9)))
print(results, f"{time.perf_counter() - start:.1f}s")    # 9 jobs / 3 at a time ≈ 1.5s

[0, 1, 2, 3, 4, 5, 6, 7, 8] 1.5s


## 7. Producer–consumer with `asyncio.Queue`

In [11]:
async def producer(q, n):
    for i in range(n):
        await q.put(f"job-{i}")
        await asyncio.sleep(0.05)

async def consumer(q, name, done):
    while True:
        job = await q.get()
        await asyncio.sleep(0.1)          # process the job
        done.append(f"{name}:{job}")
        q.task_done()

q, done = asyncio.Queue(), []
consumers = [asyncio.create_task(consumer(q, f"C{i}", done)) for i in range(3)]
await producer(q, 9)
await q.join()                            # wait until every job is processed
for c in consumers:
    c.cancel()
print(len(done), done[:4])

9 ['C0:job-0', 'C1:job-1', 'C2:job-2', 'C0:job-3']


## 8. `async for` and `async with`

In [12]:
async def ticker(n, delay=0.2):
    """An async generator."""
    for i in range(n):
        await asyncio.sleep(delay)
        yield i

async for tick in ticker(4):
    print("tick", tick)

squares = [i * i async for i in ticker(4, 0.05)]     # async comprehension
print(squares)

tick 0


tick 1


tick 2


tick 3


[0, 1, 4, 9]


In [13]:
class AsyncConnection:
    async def __aenter__(self):
        await asyncio.sleep(0.1)
        print("connected")
        return self
    async def __aexit__(self, *exc):
        await asyncio.sleep(0.1)
        print("disconnected")
    async def query(self, sql):
        await asyncio.sleep(0.1)
        return f"result of {sql!r}"

async with AsyncConnection() as conn:
    print(await conn.query("SELECT 1"))

connected


result of 'SELECT 1'


disconnected


## 9. Blocking code inside async: `asyncio.to_thread`

❌ Never call blocking functions (`time.sleep`, `requests.get`, heavy loops) directly inside a coroutine: they freeze the whole event loop. Offload them:

In [14]:
def blocking_io(n):
    time.sleep(1)                      # e.g. a library without async support
    return n * 10

start = time.perf_counter()
results = await asyncio.gather(*(asyncio.to_thread(blocking_io, i) for i in range(4)))
print(results, f"{time.perf_counter() - start:.1f}s")

[0, 10, 20, 30] 1.0s


## 10. Real HTTP with async

`requests` is synchronous. Async code uses async HTTP libraries such as **`httpx`** or **`aiohttp`**:

```python
import asyncio, httpx

async def main():
    urls = ["https://www.python.org", "https://pypi.org", "https://docs.python.org"]
    async with httpx.AsyncClient(timeout=10) as client:
        responses = await asyncio.gather(*(client.get(u) for u in urls))
    for r in responses:
        print(r.status_code, r.url)

asyncio.run(main())
```

## 11. Threads vs asyncio

| | Threads | asyncio |
|---|---|---|
| Switching | OS decides (preemptive) | at each `await` (cooperative) |
| Scale | hundreds of threads | **tens of thousands** of tasks |
| Libraries | works with any blocking library | needs async libraries (`httpx`, `asyncpg`, ...) |
| Race conditions | common (need locks) | rarer (switches only at `await`) |
| Learning curve | lower | higher (`async` spreads through code) |

> ➡️ **Continue with:** lesson 06 (event-loop internals, cancellation in depth, `shield`, synchronisation primitives, `contextvars`, eager tasks, debugging, call graphs) and lesson 07 (TCP servers & clients with streams, async subprocesses, httpx, anyio).

## 📝 Summary
- `async def` defines coroutines; `await` pauses until a result is ready.
- `asyncio.run(main())` in scripts; top-level `await` in Jupyter.
- `gather`, `create_task`, ✅ `TaskGroup` for concurrency; `asyncio.timeout` for time limits.
- `Semaphore` limits parallel work; `asyncio.Queue` for pipelines.
- Never block the event loop; use `asyncio.to_thread` for blocking calls.

## 🏋️ Practice
1. Write `countdown(name, n)` that prints a countdown with 0.5s pauses; run three countdowns concurrently with a `TaskGroup`.
2. Simulate fetching 20 URLs (random 0.1–0.5s each) with at most 5 concurrent requests; print the total time.
3. Wrap a coroutine that takes 3 s with a 1 s timeout and print a friendly message.

<details><summary>Show solutions</summary>

```python
import asyncio, random, time

async def countdown(name, n):
    for i in range(n, 0, -1):
        print(name, i)
        await asyncio.sleep(0.5)

async def main1():
    async with asyncio.TaskGroup() as tg:
        for name, n in [("A", 3), ("B", 2), ("C", 4)]:
            tg.create_task(countdown(name, n))

async def main2():
    sem = asyncio.Semaphore(5)
    async def fetch(i):
        async with sem:
            await asyncio.sleep(random.uniform(0.1, 0.5))
            return i
    t = time.perf_counter()
    await asyncio.gather(*(fetch(i) for i in range(20)))
    print(f"{time.perf_counter() - t:.2f}s")

async def main3():
    try:
        async with asyncio.timeout(1):
            await asyncio.sleep(3)
    except TimeoutError:
        print("That took too long, try again later!")

# in a script: asyncio.run(main1()); asyncio.run(main2()); asyncio.run(main3())
```
</details>